In [20]:
import pandas as pd
import numpy as np
import seaborn as sns


from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix


# ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [21]:
df = pd.read_csv('../data/pcos_prediction_dataset.csv')

In [22]:
df.head()

,Country,Age,BMI,Menstrual Regularity,Hirsutism,Acne Severity,Family History of PCOS,Insulin Resistance,Lifestyle Score,Stress Levels,Urban/Rural,Socioeconomic Status,Awareness of PCOS,Fertility Concerns,Undiagnosed PCOS Likelihood,Ethnicity,Diagnosis
0,Madagascar,26,Overweight,Regular,Yes,Severe,Yes,Yes,2,Low,Rural,High,Yes,No,0.107938,Hispanic,Yes
1,Vietnam,16,Underweight,Regular,Yes,NaN,No,Yes,4,High,Rural,Middle,Yes,No,0.156729,Other,No
2,Somalia,41,Normal,Regular,No,Moderate,No,No,7,Medium,Urban,Middle,Yes,Yes,0.202901,Other,No
3,Malawi,27,Normal,Irregular,No,Mild,No,No,10,Low,Urban,High,Yes,No,0.073926,Caucasian,Yes
4,France,26,Overweight,Irregular,Yes,NaN,No,No,7,Medium,Urban,Middle,No,No,0.229266,Caucasian,No


In [23]:
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

In [24]:
target_col = 'diagnosis'

In [25]:
# impute missing values

# Get the mode (most frequent value). mode() returns a Series, so we take the first element
mode_value = df['acne_severity'].mode()[0]

# Fill NaNs with that mode
df['acne_severity'] = df['acne_severity'].fillna(mode_value)

In [26]:
# Separate target
y = df[target_col].map({'No': 0, 'Yes': 1})

In [27]:
# Drop rows with missing target
df_clean = df.dropna(subset=[target_col]).copy()

In [ ]:
#df_clean.to_csv("df_clean.csv", index=False) # save the file

In [28]:
# 2️⃣ Identify Features
# =========================
features = df_clean.drop(columns=[target_col])
categorical_cols = features.select_dtypes(include=['string', 'category']).columns.tolist()
numerical_cols = features.select_dtypes(include=np.number).columns.tolist()


In [29]:
# 3️⃣ Encode Features
# =========================
feature_dicts = features.to_dict(orient='records')
dv = DictVectorizer(sparse=False)
X = dv.fit_transform(feature_dicts)

print("Feature names:", dv.get_feature_names_out()[:10])
print("Encoded X shape:", X.shape)

Feature names: ['acne_severity=Mild' 'acne_severity=Moderate' 'acne_severity=Severe'
 'age' 'awareness_of_pcos=No' 'awareness_of_pcos=Yes' 'bmi=Normal'
 'bmi=Obese' 'bmi=Overweight' 'bmi=Underweight']
Encoded X shape: (120000, 112)


### Prepare for Train-test-split

In [30]:
# Step 1: Split off test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Step 2: Split remaining data into train (60%) and validation (20%))
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
)

# Check shapes
print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)

Train shape: (72000, 112) (72000,)
Validation shape: (24000, 112) (24000,)
Test shape: (24000, 112) (24000,)


* Stratify each split so that the diagnosis class imbalance is maintained across train/val/test.

In [31]:
from imblearn.over_sampling import SMOTE

# 5️⃣ Handle Class Imbalance with SMOTE
# -----------------------------
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

### Train Baseline Model

In [32]:
# 5️⃣ Train Baseline Model
# =========================
model = RandomForestClassifier(random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

print("Original y_train distribution:\n", y_train.value_counts())
print("Resampled y_train distribution:\n", y_train_res.value_counts())

Original y_train distribution:
 diagnosis
0    64443
1     7557
Name: count, dtype: int64
Resampled y_train distribution:
 diagnosis
1    64443
0    64443
Name: count, dtype: int64


In [33]:
# 6️⃣ Evaluate on Validation Set
# =========================
y_val_pred = model.predict(X_val)
print("Validation Metrics:\n")
print(classification_report(y_val, y_val_pred))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))

Validation Metrics:

              precision    recall  f1-score   support

           0       0.90      1.00      0.94     21481
           1       0.00      0.00      0.00      2519

    accuracy                           0.90     24000
   macro avg       0.45      0.50      0.47     24000
weighted avg       0.80      0.90      0.85     24000

Confusion Matrix:
 [[21481     0]
 [ 2519     0]]


In [34]:
# 7️⃣ Evaluate on Test Set
# =========================
y_test_pred = model.predict(X_test)
print("Test Metrics:\n")
print(classification_report(y_test, y_test_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))

Test Metrics:

              precision    recall  f1-score   support

           0       0.90      1.00      0.94     21481
           1       0.00      0.00      0.00      2519

    accuracy                           0.90     24000
   macro avg       0.45      0.50      0.47     24000
weighted avg       0.80      0.90      0.85     24000

Confusion Matrix:
 [[21481     0]
 [ 2519     0]]


In [35]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train_res, y_train_res)

# Predict on validation set
y_val_pred = model.predict(X_val)

print("Validation Metrics:")
print(classification_report(y_val, y_val_pred))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))

Validation Metrics:
              precision    recall  f1-score   support

           0       0.90      1.00      0.94     21481
           1       0.14      0.00      0.00      2519

    accuracy                           0.89     24000
   macro avg       0.52      0.50      0.47     24000
weighted avg       0.82      0.89      0.85     24000

Confusion Matrix:
 [[21469    12]
 [ 2517     2]]


In [36]:
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1️⃣ Apply SMOTE only on training data
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:\n", y_train.value_counts())
print("After SMOTE:\n", y_train_res.value_counts())

# 2️⃣ Train RandomForest on resampled data
model = RandomForestClassifier(random_state=42)
model.fit(X_train_res, y_train_res)

# 3️⃣ Evaluate on validation set (original, not resampled)
y_val_pred = model.predict(X_val)
print("\nValidation Metrics:\n")
print(classification_report(y_val, y_val_pred))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))

Before SMOTE:
 diagnosis
0    64443
1     7557
Name: count, dtype: int64
After SMOTE:
 diagnosis
1    64443
0    64443
Name: count, dtype: int64

Validation Metrics:

              precision    recall  f1-score   support

           0       0.90      1.00      0.94     21481
           1       0.15      0.00      0.00      2519

    accuracy                           0.89     24000
   macro avg       0.52      0.50      0.47     24000
weighted avg       0.82      0.89      0.85     24000

Confusion Matrix:
 [[21470    11]
 [ 2517     2]]
